# 🌟 Modelo en Estrella y Cubo OLAP con PySpark

Este notebook carga el **modelo en estrella real desde SQL Server** (exportado a CSV en GitHub) y construye un **cubo OLAP** usando PySpark para demostrar operaciones analíticas multidimensionales.

## Fuente de datos
📦 Repositorio GitHub: https://github.com/rprieto2809/CUBOS-CSV

## Arquitectura del modelo en estrella
```
         D_Producto
              |
D_Sucursal -- H_Ventas -- D_Cliente
              |
         D_Vendedor
              |
          D_Tiempo
```

## Operaciones OLAP que veremos
| Operación | Descripción |
|-----------|-------------|
| **Roll-up** | Agregar datos (ej: de mes → trimestre → año) |
| **Drill-down** | Desagregar datos (ej: de año → mes → día) |
| **Slice** | Filtrar por una dimensión (ej: solo una categoría) |
| **Dice** | Filtrar por varias dimensiones a la vez |
| **Pivot** | Rotar el cubo para ver otra perspectiva |
| **Cube/Rollup** | Todas las agregaciones a la vez |


## ⚙️ Celda 1: Instalación y configuración de PySpark

In [ ]:
!pip install pyspark --quiet

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('CuboOLAP_ModeloEstrella') \
    .config('spark.sql.repl.eagerEval.enabled', True) \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('✅ SparkSession creada correctamente')
print(f'   Versión de Spark: {spark.version}')

## 📥 Celda 2: Carga de datos desde GitHub
Descargamos los CSV directamente desde el repositorio usando las URLs raw de GitHub.

In [ ]:
import urllib.request
import os

# URLs raw de GitHub
BASE_URL = 'https://raw.githubusercontent.com/rprieto2809/CUBOS-CSV/main/'

archivos = [
    'D_Producto.csv',
    'D_Sucursal.csv',
    'D_Cliente.csv',
    'D_Vendedor.csv',
    'D_Tiempo.csv',
    'H_Ventas.csv',
]

os.makedirs('/tmp/cubos', exist_ok=True)

for archivo in archivos:
    url = BASE_URL + archivo
    destino = f'/tmp/cubos/{archivo}'
    urllib.request.urlretrieve(url, destino)
    print(f'✅ Descargado: {archivo}')

print('\n📂 Todos los archivos descargados en /tmp/cubos/')

## 🏗️ Celda 3: Creación de las Dimensiones con PySpark

In [ ]:
# ─────────────────────────────────────────
# DIMENSIÓN: D_Producto
# ─────────────────────────────────────────
schema_producto = StructType([
    StructField('IdProducto',          IntegerType(), False),
    StructField('CodigoProducto',      StringType(),  True),
    StructField('NombreProducto',      StringType(),  True),
    StructField('DescripcionProducto', StringType(),  True),
    StructField('PrecioUnitario',      DoubleType(),  True),
    StructField('CategoriaProducto',   StringType(),  True),
])

D_Producto = spark.read.csv(
    '/tmp/cubos/D_Producto.csv',
    header=True,
    schema=schema_producto
)

# ─────────────────────────────────────────
# DIMENSIÓN: D_Sucursal
# ─────────────────────────────────────────
schema_sucursal = StructType([
    StructField('IdSucursal',          IntegerType(), False),
    StructField('CodigoSucursal',      StringType(),  True),
    StructField('NombreSucursal',      StringType(),  True),
    StructField('DireccionSucursal',   StringType(),  True),
    StructField('CiudadSucursal',      StringType(),  True),
    StructField('PaisSucursal',        StringType(),  True),
])

D_Sucursal = spark.read.csv(
    '/tmp/cubos/D_Sucursal.csv',
    header=True,
    schema=schema_sucursal
)

# ─────────────────────────────────────────
# DIMENSIÓN: D_Cliente
# ─────────────────────────────────────────
schema_cliente = StructType([
    StructField('IdCliente',           IntegerType(), False),
    StructField('CodigoCliente',       StringType(),  True),
    StructField('NombreCliente',       StringType(),  True),
    StructField('ApellidoCliente',     StringType(),  True),
    StructField('DireccionCliente',    StringType(),  True),
    StructField('CiudadCliente',       StringType(),  True),
    StructField('PaisCliente',         StringType(),  True),
])

D_Cliente = spark.read.csv(
    '/tmp/cubos/D_Cliente.csv',
    header=True,
    schema=schema_cliente
)

# ─────────────────────────────────────────
# DIMENSIÓN: D_Vendedor
# ─────────────────────────────────────────
schema_vendedor = StructType([
    StructField('IdVendedor',          IntegerType(), False),
    StructField('CodigoVendedor',      StringType(),  True),
    StructField('NombreVendedor',      StringType(),  True),
    StructField('ApellidoVendedor',    StringType(),  True),
    StructField('DireccionVendedor',   StringType(),  True),
    StructField('CiudadVendedor',      StringType(),  True),
    StructField('PaisVendedor',        StringType(),  True),
])

D_Vendedor = spark.read.csv(
    '/tmp/cubos/D_Vendedor.csv',
    header=True,
    schema=schema_vendedor
)

# ─────────────────────────────────────────
# DIMENSIÓN: D_Tiempo
# ─────────────────────────────────────────
schema_tiempo = StructType([
    StructField('IdTiempo',    IntegerType(), False),
    StructField('Fecha',       DateType(),    True),
    StructField('Anio',        IntegerType(), True),
    StructField('Mes',         IntegerType(), True),
    StructField('Dia',         IntegerType(), True),
    StructField('Trimestre',   IntegerType(), True),
    StructField('Semestre',    IntegerType(), True),
])

D_Tiempo = spark.read.csv(
    '/tmp/cubos/D_Tiempo.csv',
    header=True,
    schema=schema_tiempo
)

# ─────────────────────────────────────────
# TABLA DE HECHOS: H_Ventas
# ─────────────────────────────────────────
schema_ventas = StructType([
    StructField('IdVenta',       IntegerType(), False),
    StructField('IdProducto',    IntegerType(), True),
    StructField('IdSucursal',    IntegerType(), True),
    StructField('IdCliente',     IntegerType(), True),
    StructField('IdVendedor',    IntegerType(), True),
    StructField('IdTiempo',      IntegerType(), True),
    StructField('CantidadVenta', IntegerType(), True),
    StructField('TotalVenta',    DoubleType(),  True),
])

H_Ventas = spark.read.csv(
    '/tmp/cubos/H_Ventas.csv',
    header=True,
    schema=schema_ventas
)

print('✅ Tablas cargadas desde GitHub:')
print(f'   D_Producto  → {D_Producto.count()} registros')
print(f'   D_Sucursal  → {D_Sucursal.count()} registros')
print(f'   D_Cliente   → {D_Cliente.count()} registros')
print(f'   D_Vendedor  → {D_Vendedor.count()} registros')
print(f'   D_Tiempo    → {D_Tiempo.count()} registros')
print(f'   H_Ventas    → {H_Ventas.count()} registros')

## 🔎 Celda 4: Verificación de los datos cargados
Comprobamos que las tablas se han cargado correctamente antes de construir el cubo.

In [ ]:
print('── D_Producto ──')
D_Producto.show(3, truncate=40)

print('── D_Sucursal ──')
D_Sucursal.show(3, truncate=40)

print('── D_Tiempo ──')
D_Tiempo.show(3)

print('── H_Ventas ──')
H_Ventas.show(5)

## 🔗 Celda 5: Construcción del Cubo
Unimos la tabla de hechos con todas las dimensiones. Este DataFrame desnormalizado es la base del cubo OLAP — desde aquí podemos analizar los datos desde cualquier perspectiva.

In [ ]:
Cubo_Ventas = H_Ventas \
    .join(D_Producto,  on='IdProducto',  how='left') \
    .join(D_Sucursal,  on='IdSucursal',  how='left') \
    .join(D_Cliente,   on='IdCliente',   how='left') \
    .join(D_Vendedor,  on='IdVendedor',  how='left') \
    .join(D_Tiempo,    on='IdTiempo',    how='left') \
    .select(
        'IdVenta',
        # Dimensión Tiempo
        'Fecha', 'Anio', 'Trimestre', 'Semestre', 'Mes', 'Dia',
        # Dimensión Producto
        'NombreProducto', 'CategoriaProducto', 'PrecioUnitario',
        # Dimensión Sucursal
        'NombreSucursal', 'CiudadSucursal', 'PaisSucursal',
        # Dimensión Cliente
        F.concat(F.col('NombreCliente'), F.lit(' '), F.col('ApellidoCliente')).alias('NombreCliente'),
        'CiudadCliente', 'PaisCliente',
        # Dimensión Vendedor
        F.concat(F.col('NombreVendedor'), F.lit(' '), F.col('ApellidoVendedor')).alias('NombreVendedor'),
        'CiudadVendedor', 'PaisVendedor',
        # Métricas
        'CantidadVenta', 'TotalVenta'
    )

# Registramos el cubo como vista SQL
Cubo_Ventas.createOrReplaceTempView('Cubo_Ventas')

print(f'✅ Cubo construido: {Cubo_Ventas.count()} filas × {len(Cubo_Ventas.columns)} columnas')
print()
print('Columnas disponibles:')
print(Cubo_Ventas.columns)
print()
Cubo_Ventas.show(3, truncate=30)

## 🔺 Celda 6: ROLL-UP — Agregar hacia niveles superiores
**Roll-up** = subir en la jerarquía agrupando más datos.
Ejemplo: Día → Mes → Trimestre → Año

In [ ]:
print('=' * 60)
print('🔺 ROLL-UP: Mes → Trimestre → Año')
print('=' * 60)

print('\n📅 Nivel MES (más detalle):')
spark.sql('''
    SELECT
        Anio,
        Mes,
        ROUND(SUM(TotalVenta), 2)  AS TotalVentas,
        SUM(CantidadVenta)         AS Unidades,
        COUNT(*)                   AS Transacciones
    FROM Cubo_Ventas
    GROUP BY Anio, Mes
    ORDER BY Anio, Mes
''').show(6)

print('\n📅 Nivel TRIMESTRE (roll-up x1):')
spark.sql('''
    SELECT
        Anio,
        CONCAT('Q', Trimestre)     AS Trimestre,
        ROUND(SUM(TotalVenta), 2)  AS TotalVentas,
        SUM(CantidadVenta)         AS Unidades,
        COUNT(*)                   AS Transacciones
    FROM Cubo_Ventas
    GROUP BY Anio, Trimestre
    ORDER BY Anio, Trimestre
''').show()

print('\n📅 Nivel AÑO (roll-up x2 — máxima agregación):')
spark.sql('''
    SELECT
        Anio,
        ROUND(SUM(TotalVenta), 2)  AS TotalVentas,
        SUM(CantidadVenta)         AS Unidades,
        COUNT(*)                   AS Transacciones,
        ROUND(AVG(TotalVenta), 2)  AS TicketMedio
    FROM Cubo_Ventas
    GROUP BY Anio
    ORDER BY Anio
''').show()

## 🔻 Celda 7: DRILL-DOWN — Desagregar para ver más detalle
**Drill-down** = bajar en la jerarquía para ver más granularidad.
Ejemplo: País → Ciudad → Sucursal

In [ ]:
print('=' * 60)
print('🔻 DRILL-DOWN: País → Ciudad → Sucursal')
print('=' * 60)

print('\n🌍 Nivel PAÍS:')
spark.sql('''
    SELECT
        PaisSucursal               AS Pais,
        ROUND(SUM(TotalVenta), 2)  AS TotalVentas,
        SUM(CantidadVenta)         AS Unidades
    FROM Cubo_Ventas
    GROUP BY PaisSucursal
    ORDER BY TotalVentas DESC
''').show()

print('\n🏙️ Nivel CIUDAD:')
spark.sql('''
    SELECT
        PaisSucursal               AS Pais,
        CiudadSucursal             AS Ciudad,
        ROUND(SUM(TotalVenta), 2)  AS TotalVentas,
        SUM(CantidadVenta)         AS Unidades
    FROM Cubo_Ventas
    GROUP BY PaisSucursal, CiudadSucursal
    ORDER BY TotalVentas DESC
''').show()

print('\n🏪 Nivel SUCURSAL + CATEGORÍA (máximo detalle):')
spark.sql('''
    SELECT
        NombreSucursal,
        CategoriaProducto,
        ROUND(SUM(TotalVenta), 2)  AS TotalVentas,
        SUM(CantidadVenta)         AS Unidades,
        COUNT(*)                   AS Transacciones
    FROM Cubo_Ventas
    GROUP BY NombreSucursal, CategoriaProducto
    ORDER BY NombreSucursal, TotalVentas DESC
''').show(30, truncate=30)

## 🔪 Celda 8: SLICE — Cortar el cubo por una dimensión
**Slice** = fijamos el valor de **una** dimensión y analizamos el resto.
Como cortar una rebanada del cubo.

In [ ]:
print('=' * 60)
print('🔪 SLICE: Una categoría de producto concreta')
print('=' * 60)

# Primero vemos qué categorías existen
print('\n📦 Categorías disponibles:')
Cubo_Ventas.select('CategoriaProducto').distinct().orderBy('CategoriaProducto').show()

# Slice: tomamos la primera categoría del cubo real
categoria_slice = Cubo_Ventas.select('CategoriaProducto').distinct().orderBy('CategoriaProducto').first()[0]
print(f'\n🔪 Slice aplicado → Categoría: "{categoria_slice}"')

slice_cat = Cubo_Ventas.filter(F.col('CategoriaProducto') == categoria_slice)
print(f'Registros en el slice: {slice_cat.count()}')

print('\n📊 Ventas por Sucursal en esta categoría:')
slice_cat.groupBy('NombreSucursal') \
    .agg(
        F.round(F.sum('TotalVenta'), 2).alias('TotalVentas'),
        F.sum('CantidadVenta').alias('Unidades'),
        F.count('*').alias('Transacciones')
    ) \
    .orderBy(F.desc('TotalVentas')) \
    .show(truncate=30)

print('\n📅 Evolución mensual del slice:')
slice_cat.groupBy('Anio', 'Mes') \
    .agg(F.round(F.sum('TotalVenta'), 2).alias('TotalVentas')) \
    .orderBy('Anio', 'Mes') \
    .show()

## 🎲 Celda 9: DICE — Cortar el cubo por varias dimensiones
**Dice** = fijamos valores en **varias** dimensiones simultáneamente.
Como extraer un subcubo del cubo original.

In [ ]:
print('=' * 60)
print('🎲 DICE: Varias dimensiones a la vez')
print('=' * 60)

# Tomamos valores reales del cubo
pais     = Cubo_Ventas.select('PaisSucursal').distinct().orderBy('PaisSucursal').first()[0]
categoria = Cubo_Ventas.select('CategoriaProducto').distinct().orderBy('CategoriaProducto').first()[0]
anio     = Cubo_Ventas.select('Anio').distinct().orderBy('Anio').first()[0]

print(f'\n🎲 Filtros aplicados:')
print(f'   País:      {pais}')
print(f'   Categoría: {categoria}')
print(f'   Año:       {anio}')

dice = Cubo_Ventas.filter(
    (F.col('PaisSucursal')      == pais)      &
    (F.col('CategoriaProducto') == categoria) &
    (F.col('Anio')              == anio)
)

print(f'\nRegistros en el subcubo: {dice.count()}')

print('\n📊 Ventas por Ciudad y Producto (subcubo):')
dice.groupBy('CiudadSucursal', 'NombreProducto') \
    .agg(
        F.round(F.sum('TotalVenta'), 2).alias('TotalVentas'),
        F.sum('CantidadVenta').alias('Unidades')
    ) \
    .orderBy('CiudadSucursal', F.desc('TotalVentas')) \
    .show(20, truncate=30)

print('\n📅 Evolución mensual del subcubo:')
dice.groupBy('Mes') \
    .agg(F.round(F.sum('TotalVenta'), 2).alias('TotalVentas'),
         F.sum('CantidadVenta').alias('Unidades')) \
    .orderBy('Mes') \
    .show()

## 🔄 Celda 10: PIVOT — Tabla cruzada (rotación del cubo)
**Pivot** = reorganizamos los datos poniendo los valores de una dimensión como columnas.
Típica tabla cruzada: filas = tiempo, columnas = categoría.

In [ ]:
print('=' * 60)
print('🔄 PIVOT: Ventas por Trimestre × Categoría')
print('=' * 60)

# Obtenemos las categorías reales del cubo
categorias = [r[0] for r in Cubo_Ventas.select('CategoriaProducto').distinct().orderBy('CategoriaProducto').collect()]
print(f'\nCategorías detectadas: {categorias}')

print('\n💰 Ventas (€) por Trimestre × Categoría:')
Cubo_Ventas \
    .groupBy('Anio', 'Trimestre') \
    .pivot('CategoriaProducto', categorias) \
    .agg(F.round(F.sum('TotalVenta'), 2)) \
    .orderBy('Anio', 'Trimestre') \
    .show(truncate=15)

# Pivot: País × Año
anios = [r[0] for r in Cubo_Ventas.select('Anio').distinct().orderBy('Anio').collect()]
print(f'\n🌍 PIVOT: Ventas por País × Año {anios}:')
pivot_pais = Cubo_Ventas \
    .groupBy('PaisSucursal') \
    .pivot('Anio', anios) \
    .agg(F.round(F.sum('TotalVenta'), 2)) \
    .orderBy(F.desc(str(anios[-1])))

pivot_pais.show()

# Variación interanual si hay más de un año
if len(anios) >= 2:
    col_a1 = str(anios[0])
    col_a2 = str(anios[1])
    print(f'\n📈 Variación interanual {col_a1} vs {col_a2}:')
    pivot_pais \
        .withColumn('Variacion_%',
            F.round(((F.col(col_a2) - F.col(col_a1)) / F.col(col_a1)) * 100, 1)
        ).show()

## 📊 Celda 11: CUBE / ROLLUP — Todos los niveles a la vez
PySpark soporta `ROLLUP` y `CUBE` en SQL, operaciones nativas de los cubos OLAP que generan **todos los niveles de agregación en una sola consulta**.

In [ ]:
print('=' * 60)
print('📊 ROLLUP — Jerarquía geográfica completa')
print('=' * 60)

print('\n🏗️ ROLLUP: País → Ciudad → Sucursal (con subtotales automáticos):')
spark.sql('''
    SELECT
        COALESCE(PaisSucursal,   '📦 TOTAL GLOBAL')   AS Pais,
        COALESCE(CiudadSucursal, '  └─ Subtotal')     AS Ciudad,
        COALESCE(NombreSucursal, '     └─ Subtotal')  AS Sucursal,
        ROUND(SUM(TotalVenta), 2)                     AS TotalVentas,
        SUM(CantidadVenta)                            AS Unidades
    FROM Cubo_Ventas
    GROUP BY ROLLUP(PaisSucursal, CiudadSucursal, NombreSucursal)
    ORDER BY Pais, Ciudad, Sucursal
''').show(40, truncate=25)

print('\n🎲 CUBE: Todas las combinaciones Año × País × Categoría:')
spark.sql('''
    SELECT
        COALESCE(CAST(Anio AS STRING), 'TODOS')  AS Anio,
        COALESCE(PaisSucursal,         'TODOS')  AS Pais,
        COALESCE(CategoriaProducto,    'TODAS')  AS Categoria,
        ROUND(SUM(TotalVenta), 2)                AS TotalVentas,
        SUM(CantidadVenta)                       AS Unidades
    FROM Cubo_Ventas
    GROUP BY CUBE(Anio, PaisSucursal, CategoriaProducto)
    ORDER BY Anio DESC, Pais, Categoria
''').show(40, truncate=20)

## 🏆 Celda 12: Window Functions — Análisis avanzado
Funciones de ventana para rankings, acumulados y cuotas de mercado.

In [ ]:
print('=' * 60)
print('🏆 WINDOW FUNCTIONS')
print('=' * 60)

print('\n🥇 Ranking de productos por categoría:')
spark.sql('''
    SELECT
        CategoriaProducto,
        NombreProducto,
        ROUND(SUM(TotalVenta), 2)  AS TotalVentas,
        RANK() OVER (
            PARTITION BY CategoriaProducto
            ORDER BY SUM(TotalVenta) DESC
        )                          AS RankEnCategoria
    FROM Cubo_Ventas
    GROUP BY CategoriaProducto, NombreProducto
    ORDER BY CategoriaProducto, RankEnCategoria
''').show(30, truncate=30)

print('\n📈 Ventas acumuladas mes a mes (YTD) por año:')
spark.sql('''
    WITH ventas_mes AS (
        SELECT Anio, Mes,
               ROUND(SUM(TotalVenta), 2) AS VentasMes
        FROM Cubo_Ventas
        GROUP BY Anio, Mes
    )
    SELECT
        Anio, Mes, VentasMes,
        ROUND(SUM(VentasMes) OVER (
            PARTITION BY Anio
            ORDER BY Mes
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2) AS VentasAcumuladas_YTD
    FROM ventas_mes
    ORDER BY Anio, Mes
''').show(30)

print('\n📊 Cuota de mercado por País:')
spark.sql('''
    SELECT
        PaisSucursal,
        ROUND(SUM(TotalVenta), 2)   AS TotalVentas,
        ROUND(
            SUM(TotalVenta) * 100.0 / SUM(SUM(TotalVenta)) OVER (), 2
        )                           AS PorcentajeMercado
    FROM Cubo_Ventas
    GROUP BY PaisSucursal
    ORDER BY TotalVentas DESC
''').show()

print('\n🏅 Ranking de vendedores por total vendido:')
spark.sql('''
    SELECT
        RANK() OVER (ORDER BY SUM(TotalVenta) DESC) AS Ranking,
        NombreVendedor,
        COUNT(*)                    AS NumVentas,
        SUM(CantidadVenta)          AS UnidadesVendidas,
        ROUND(SUM(TotalVenta), 2)   AS TotalVentas,
        ROUND(AVG(TotalVenta), 2)   AS TicketMedio
    FROM Cubo_Ventas
    GROUP BY NombreVendedor
    ORDER BY Ranking
''').show(truncate=30)

## 📋 Celda 13: Resumen final

| Operación | ¿Qué hace? | Ejemplo en este notebook |
|-----------|-----------|-------------------------|
| **Roll-up** | Agrupa hacia niveles superiores | Mes → Trimestre → Año |
| **Drill-down** | Desagrega hacia más detalle | País → Ciudad → Sucursal |
| **Slice** | Filtra por **una** dimensión | Solo una categoría |
| **Dice** | Filtra por **varias** dimensiones | País + Categoría + Año |
| **Pivot** | Rota dimensiones como columnas | Trimestre × Categoría |
| **Cube/Rollup** | Genera todas las agregaciones | País + Ciudad + Sucursal |
| **Window** | Rankings y acumulados | Ranking vendedores, YTD |

In [ ]:
total_ventas  = Cubo_Ventas.agg(F.round(F.sum('TotalVenta'), 2)).collect()[0][0]
total_uds     = Cubo_Ventas.agg(F.sum('CantidadVenta')).collect()[0][0]
total_trans   = Cubo_Ventas.count()

print('✅ Notebook completado.')
print()
print('Resumen del cubo:')
print(f'  Total ventas:        {total_ventas} €')
print(f'  Total unidades:      {total_uds}')
print(f'  Total transacciones: {total_trans}')
print()
spark.stop()
print('🔴 SparkSession cerrada.')